In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
from birddog.database import Database

2026-01-18 11:35:22,618 [INFO] Using local nocodb api: http://localhost:8080


In [3]:
db = Database()

2026-01-18 11:35:22,620 [INFO] GET: http://localhost:8080/api/v2/meta/bases/p79fvr9cjqgpv5n/tables, None
2026-01-18 11:35:22,643 [INFO] GET: http://localhost:8080/api/v2/tables/mlenpssyo8w1q91/records, {'offset': 0, 'limit': 100}
2026-01-18 11:35:22,672 [INFO] GET: http://localhost:8080/api/v2/tables/mbtdqbcpoff1rhx/records, {'offset': 0, 'limit': 100}


In [4]:
nocodb_aws_host = os.environ["BIRDDOG_AWS_NOCODB_HOST"]
nocodb_aws_token = os.environ["BIRDDOG_AWS_NOCODB_API_TOKEN"]
nocodb_aws_base_id = os.environ["BIRDDOG_AWS_BASE_ID"]

In [5]:
#nocodb_aws_host = nocodb_aws_host.replace("https", "http")
nocodb_aws_host

'http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com'

In [6]:
db2 = Database(host=nocodb_aws_host, api_token=nocodb_aws_token, base_id=nocodb_aws_base_id)

2026-01-18 11:35:29,210 [INFO] GET: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/meta/bases/p9dphc780lvanoc/tables, None
2026-01-18 11:35:29,360 [WARNING] No schema found for database. Running schema-less.


In [7]:
#def _list_tables_url():
#    return f"{_NOCODB_V2_API_ROOT}/meta/bases/{_NOCODB_BASE_ID}/tables"
#def _table_info_url(table_id):
#    return f"{_NOCODB_V2_API_ROOT}/meta/tables/{table_id}" 

In [8]:
def clone_table_schema(db1, table_name1, db2, table_name2):
    table_id1 = db1._table_id(table_name1)
    if db2._valid_table_name(table_name2):
        raise ValueError(f"cannot clone to existing table: {table_name2}")
    info = db1._fetch(db1._table_info_url(table_id1))
    columns = []
    for i, c in enumerate(info["columns"]):
        if not c["system"] and c["uidt"] not in ("Links", "ForeignKey", "Formula",):
            spec = {
                "title": c["title"],
                "description": c["description"],
                "uidt": c["uidt"],
            }
            if spec["uidt"] in ("SingleSelect", "MultiSelect"):
                spec["colOptions"] = {
                    "options": [
                        { "title": option["title"], "color": option["color"] }
                        for option in c["colOptions"]["options"]
                    ]
                }
            columns.append(spec)        
    create_spec = {
        "title": table_name2,
        "description": info["description"],
        "columns": columns,
    }
    db2._fetch(db2._list_tables_url(), json=create_spec, method="POST")
    return create_spec

In [9]:
clone_table_schema(db, "Schema", db2, "Schema")

2026-01-18 11:35:48,691 [INFO] GET: http://localhost:8080/api/v2/meta/tables/mlenpssyo8w1q91, None
2026-01-18 11:35:48,719 [INFO] POST: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/meta/bases/p9dphc780lvanoc/tables, None


{'title': 'Schema',
 'description': None,
 'columns': [{'title': 'Id', 'description': None, 'uidt': 'ID'},
  {'title': 'table_name', 'description': None, 'uidt': 'SingleLineText'},
  {'title': 'field_name', 'description': None, 'uidt': 'SingleLineText'},
  {'title': 'field_type',
   'description': None,
   'uidt': 'SingleSelect',
   'colOptions': {'options': [{'title': 'bool', 'color': '#cfdffe'},
     {'title': 'number', 'color': '#d0f1fd'},
     {'title': 'text', 'color': '#c2f5e8'},
     {'title': 'single_select', 'color': '#ffdaf6'},
     {'title': 'url', 'color': '#ffdce5'},
     {'title': 'multi_select', 'color': '#fee2d5'},
     {'title': 'user', 'color': '#ffeab6'},
     {'title': 'date', 'color': '#d1f7c4'},
     {'title': 'link', 'color': '#ede2fe'},
     {'title': 'links', 'color': '#cfdffe'}]}},
  {'title': 'key_field', 'description': None, 'uidt': 'Checkbox'},
  {'title': 'description', 'description': None, 'uidt': 'SingleLineText'}]}

In [10]:
clone_table_schema(db, "Schema Values", db2, "Schema Values")

2026-01-18 11:36:59,039 [INFO] GET: http://localhost:8080/api/v2/meta/tables/mbtdqbcpoff1rhx, None
2026-01-18 11:36:59,067 [INFO] POST: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/meta/bases/p9dphc780lvanoc/tables, None


{'title': 'Schema Values',
 'description': None,
 'columns': [{'title': 'Id', 'description': None, 'uidt': 'ID'},
  {'title': 'table_name', 'description': None, 'uidt': 'SingleLineText'},
  {'title': 'field_name', 'description': None, 'uidt': 'SingleLineText'},
  {'title': 'field_value', 'description': None, 'uidt': 'SingleLineText'},
  {'title': 'description', 'description': None, 'uidt': 'SingleLineText'}]}

In [11]:
db2 = Database(host=nocodb_aws_host, api_token=nocodb_aws_token, base_id=nocodb_aws_base_id)

2026-01-18 11:37:39,667 [INFO] GET: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/meta/bases/p9dphc780lvanoc/tables, None
2026-01-18 11:37:39,817 [INFO] GET: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/tables/mknlfk5rf9l0xkn/records, {'offset': 0, 'limit': 100}
2026-01-18 11:37:39,889 [INFO] GET: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/tables/m63lnet45jqwswf/records, {'offset': 0, 'limit': 100}


In [12]:
def copy_records(db1, table1, db2, table2):
    while True:
        records, cursor = db1.scan(table1)
        records = db1.encode_records(table1, records)
        db2.write(table2, records, raw=True)
        if not cursor:
            break

In [13]:
copy_records(db, "Schema Values", db2, "Schema Values")

2026-01-18 11:38:19,531 [INFO] GET: http://localhost:8080/api/v2/tables/mbtdqbcpoff1rhx/records, {'offset': 0, 'limit': 100}
2026-01-18 11:38:19,565 [INFO] POST: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/tables/m63lnet45jqwswf/records, None
2026-01-18 11:38:19,649 [INFO] POST: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/tables/m63lnet45jqwswf/records, None
2026-01-18 11:38:19,730 [INFO] POST: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/tables/m63lnet45jqwswf/records, None
2026-01-18 11:38:19,809 [INFO] POST: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/tables/m63lnet45jqwswf/records, None


In [14]:
copy_records(db, "Schema", db2, "Schema")

2026-01-18 11:39:22,856 [INFO] GET: http://localhost:8080/api/v2/tables/mlenpssyo8w1q91/records, {'offset': 0, 'limit': 100}
2026-01-18 11:39:22,884 [INFO] POST: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/tables/mknlfk5rf9l0xkn/records, None
2026-01-18 11:39:22,963 [INFO] POST: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/tables/mknlfk5rf9l0xkn/records, None
2026-01-18 11:39:23,047 [INFO] POST: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/tables/mknlfk5rf9l0xkn/records, None
2026-01-18 11:39:23,128 [INFO] POST: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/tables/mknlfk5rf9l0xkn/records, None
2026-01-18 11:39:23,210 [INFO] POST: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/tables/mknlfk5rf9l0xkn/records, None


In [15]:
clone_table_schema(db, "Pages", db2, "Pages")

2026-01-18 11:40:01,431 [INFO] GET: http://localhost:8080/api/v2/meta/tables/msnw0n3dk4epz9f, None
2026-01-18 11:40:01,464 [INFO] POST: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/meta/bases/p9dphc780lvanoc/tables, None


{'title': 'Pages',
 'description': '',
 'columns': [{'title': 'Id', 'description': None, 'uidt': 'ID'},
  {'title': 'title', 'description': None, 'uidt': 'SingleLineText'},
  {'title': 'description', 'description': None, 'uidt': 'SingleLineText'},
  {'title': 'description_uk', 'description': None, 'uidt': 'SingleLineText'},
  {'title': 'availability',
   'description': None,
   'uidt': 'SingleSelect',
   'colOptions': {'options': [{'title': 'linked', 'color': '#92B7FFFF'},
     {'title': 'unlinked', 'color': '#929292FF'},
     {'title': 'redlinked', 'color': '#F5C4C2FF'}]}},
  {'title': 'comments', 'description': None, 'uidt': 'SingleLineText'},
  {'title': 'level',
   'description': None,
   'uidt': 'SingleSelect',
   'colOptions': {'options': [{'title': 'archive', 'color': '#cfdffe'},
     {'title': 'fond', 'color': '#d0f1fd'},
     {'title': 'opus', 'color': '#c2f5e8'},
     {'title': 'case', 'color': '#ffdaf6'}]}},
  {'title': 'change_date', 'description': None, 'uidt': 'DateTime'}

In [16]:
clone_table_schema(db, "Documents", db2, "Documents")

2026-01-18 11:40:04,820 [INFO] GET: http://localhost:8080/api/v2/meta/tables/mi63ruftme2ph8l, None
2026-01-18 11:40:04,853 [INFO] POST: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/meta/bases/p9dphc780lvanoc/tables, None


{'title': 'Documents',
 'description': None,
 'columns': [{'title': 'Id', 'description': None, 'uidt': 'ID'},
  {'title': 'title', 'description': None, 'uidt': 'SingleLineText'},
  {'title': 'link', 'description': None, 'uidt': 'URL'},
  {'title': 'processor', 'description': None, 'uidt': 'SingleLineText'},
  {'title': 'pages_processed', 'description': None, 'uidt': 'Number'},
  {'title': 'doc_type',
   'description': None,
   'uidt': 'MultiSelect',
   'colOptions': {'options': [{'title': 'O', 'color': '#cfdffe'},
     {'title': 'L', 'color': '#d0f1fd'},
     {'title': 'O/L', 'color': '#c2f5e8'},
     {'title': 'C', 'color': '#ffdaf6'},
     {'title': 'V', 'color': '#ffdce5'},
     {'title': 'O/V', 'color': '#fee2d5'}]}},
  {'title': 'content_code',
   'description': None,
   'uidt': 'SingleSelect',
   'colOptions': {'options': [{'title': 'U', 'color': '#cfdffe'},
     {'title': 'J', 'color': '#d0f1fd'},
     {'title': 'N', 'color': '#c2f5e8'},
     {'title': 'M', 'color': '#ffdaf6'},


In [19]:
db2 = Database(host=nocodb_aws_host, api_token=nocodb_aws_token, base_id=nocodb_aws_base_id)

2026-01-18 11:52:30,694 [INFO] GET: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/meta/bases/p9dphc780lvanoc/tables, None
2026-01-18 11:52:30,844 [INFO] GET: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/tables/mknlfk5rf9l0xkn/records, {'offset': 0, 'limit': 100}
2026-01-18 11:52:30,922 [INFO] GET: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/tables/m63lnet45jqwswf/records, {'offset': 0, 'limit': 100}


In [21]:
def rename_field(db, field_id, new_name):
    url = f"{db._host}/api/v2/meta/columns/{field_id}"
    payload = { "title": new_name }
    db._fetch(url, json=payload, method="PATCH")

In [22]:
db2._field_id("Pages", "Pages")

'csbynq0fiamol79'

In [23]:
rename_field(db2, 'csbynq0fiamol79', "parent")

2026-01-18 11:53:58,128 [INFO] PATCH: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com/api/v2/meta/columns/csbynq0fiamol79, None
